# A/B Test Statistical Significance

**Author:** Yuliia Butenko

**Goal:** analyse the results of four A/B tests with statistical methods in Python and visualise the key conversion metrics in Tableau.

**Links:**
- 📊 Tableau dashboard: [A/B Test Dashboard](https://public.tableau.com/views/ABTestDashboard_17925770139850/Significance)
- 📁 CSV with calculation results: [ab_test_results.csv](https://github.com/ButenkoYuliia/ab-test-significance/blob/main/data/ab_test_results.csv)

## Project steps
1. Load and check the data (export from BigQuery, "DA" dataset)
2. Calculate statistical significance for 4 metrics — overall and by segment (device, continent, country, channel)
3. Save the results to CSV for Tableau
4. Visualise in Tableau
5. Conclusions

## 1. Load data

The data is an aggregated export from BigQuery: each row contains the date, test, group, segments (country, device, continent, channel), the event name and the number of **unique sessions** with that event (`value`).

👉 Before running, set `DATA_PATH` to your file location.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/ab_test_data.csv'   # ← replace with your path

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## 2. Data check

Check the event names and make sure the number of sessions with an event **never exceeds** the total number of sessions (otherwise the conversion rate would be > 100 % and the test would be invalid).

In [ ]:
print(df['event_name'].unique())

check = (df.groupby(['test', 'test_group', 'event_name'])['value']
           .sum().unstack('event_name'))

cols = ['session', 'add_payment_info', 'add_shipping_info', 'begin_checkout', 'new accounts']
display(check[cols])

# conversion rate in %, all values must be below 100
display((check[cols[1:]].div(check['session'], axis=0) * 100).round(2))

## 3. Metric settings

Only the names of the 4 metrics and how they are calculated (event / sessions) are hard-coded. Everything else — tests, groups, segments — is taken from the data automatically inside loops. Adding a new metric takes one line in `METRICS`; adding a new segment takes one word in `SEGMENTS`.

In [ ]:
DENOM = 'session'                      # denominator for all metrics
METRICS = {                            # metric name : numerator event
    'add_payment_info / session':  'add_payment_info',
    'add_shipping_info / session': 'add_shipping_info',
    'begin_checkout / session':    'begin_checkout',
    'new_accounts / session':      'new accounts',
}
SEGMENTS = ['total', 'device', 'continent', 'country', 'channel']
CONTROL, TREATMENT = 1, 2              # 1 = control group, 2 = test group
ALPHA = 0.05                           # significance level

df['total'] = 'Total'                  # artificial segment for the overall (total) calculation
EVENTS = list(METRICS.values()) + [DENOM]
print('Settings ready')

## 4. Statistical significance

**Logic:**
- Each metric is the **share of sessions** with the event (conversion rate): `CR = sessions with event / all sessions`.
- Conversion rates of the control and test groups are compared with a **two-proportion z-test** (`proportions_ztest` from statsmodels).
- Null hypothesis H₀: the conversion rates of both groups are equal.
- If **p-value < 0.05**, H₀ is rejected → the difference is **statistically significant**:
  - test CR higher → `Significant increase`
  - test CR lower → `Significant decrease`
- If p-value ≥ 0.05 → `Not significant`.

The calculation runs in three nested loops: **segment → (test, segment value) → metric**.

In [ ]:
results = []

for seg in SEGMENTS:
    # sum of sessions per event for each test / segment value / group
    agg = (df[df['event_name'].isin(EVENTS)]
           .groupby(['test', seg, 'test_group', 'event_name'])['value'].sum()
           .unstack('event_name', fill_value=0))

    for (test, seg_value), g in agg.groupby(level=[0, 1]):
        g = g.droplevel([0, 1])
        if not {CONTROL, TREATMENT}.issubset(g.index):   # both groups are required
            continue
        ctrl, trt = g.loc[CONTROL], g.loc[TREATMENT]

        for metric, event in METRICS.items():
            n1, n2 = ctrl.get(DENOM, 0), trt.get(DENOM, 0)   # sessions
            x1, x2 = ctrl.get(event, 0), trt.get(event, 0)   # sessions with the event
            if n1 == 0 or n2 == 0:
                continue

            cr1, cr2 = x1 / n1, x2 / n2
            z, p = proportions_ztest([x2, x1], [n2, n1])
            if np.isnan(p):          # both conversions = 0 → no difference
                p = 1.0

            significant = p < ALPHA
            if not significant:
                label = 'Not significant'
            else:
                label = 'Significant increase' if cr2 > cr1 else 'Significant decrease'

            results.append({
                'test': test,
                'segment_type': seg,
                'segment_value': seg_value,
                'metric': metric,
                'sessions_control': n1, 'sessions_test': n2,
                'events_control': x1,   'events_test': x2,
                'cr_control': cr1,      'cr_test': cr2,
                'uplift_pct': (cr2 - cr1) / cr1 * 100 if cr1 > 0 else np.nan,
                'p_value': p,
                'is_significant': significant,
                'result': label,
            })

res = pd.DataFrame(results)
print('Total result rows:', len(res))

## 5. Overall results per test

In [ ]:
total = res[res['segment_type'] == 'total'].copy()
view = total[['test', 'metric', 'cr_control', 'cr_test', 'uplift_pct', 'p_value', 'result']].copy()
view['cr_control'] = (view['cr_control'] * 100).round(2).astype(str) + ' %'
view['cr_test']    = (view['cr_test'] * 100).round(2).astype(str) + ' %'
view['uplift_pct'] = view['uplift_pct'].round(2).astype(str) + ' %'
view['p_value']    = view['p_value'].round(4)
display(view.reset_index(drop=True))

## 6. Significant results by segment

Number of significant changes per segment type. ⚠️ Results in small segments (e.g. countries with a few dozen sessions) are unreliable, so a minimum-sessions filter is applied.

In [ ]:
MIN_SESSIONS = 1000   # minimum sessions per group to trust the result
res['enough_data'] = (res['sessions_control'] >= MIN_SESSIONS) & (res['sessions_test'] >= MIN_SESSIONS)

summary = (res[res['enough_data']]
           .groupby(['segment_type', 'result']).size()
           .unstack(fill_value=0))
display(summary)

sig = res[res['is_significant'] & res['enough_data'] & (res['segment_type'] != 'total')]
display(sig[['test', 'segment_type', 'segment_value', 'metric',
             'cr_control', 'cr_test', 'uplift_pct', 'p_value', 'result']]
        .sort_values('p_value').reset_index(drop=True))

## 7. Save results for Tableau

In [ ]:
OUTPUT_PATH = '/content/drive/MyDrive/ab_test_results.csv'
res.to_csv(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)

## 8. Conclusions

### Overall results

| Test | Metric | Control | Test | Uplift | p-value | Result |
|---|---|---|---|---|---|---|
| 1 | add_shipping_info / session | 4.10 % | 4.41 % | +7.5 % | 0.022 | ✅ significant increase |
| 1 | begin_checkout / session | 4.10 % | 4.41 % | +7.4 % | 0.024 | ✅ significant increase |
| 4 | new_accounts / session | 8.55 % | 8.26 % | −3.4 % | 0.018 | 🔻 significant decrease |
| other 13 checks | — | — | — | −3.4 % to +3.4 % | > 0.05 | not significant |

- **Test 1 — positive result.** Conversion to `add_shipping_info` and `begin_checkout` rose from 4.10 % to 4.41 % (≈ +7.5 %, p ≈ 0.02), a significant improvement at the start of checkout. `add_payment_info` (+2.4 %, p = 0.60) and `new_accounts` (−3.4 %, p = 0.12) did not change significantly, so the gain does not yet reach the payment step.
- **Test 2 — no effect.** All 4 metrics changed within ±3.5 %, p-value > 0.4.
- **Test 3 — no effect.** Changes below ±1.5 %, p-value > 0.5.
- **Test 4 — negative result.** Conversion to `new_accounts` fell from 8.55 % to 8.26 % (−3.4 %, p ≈ 0.02), a significant drop in sign-ups. Other metrics did not change (p > 0.8).

### By segment
(only segments with ≥ 1,000 sessions in each group)

- **Test 1:** the increase in `add_shipping_info` and `begin_checkout` is strongest on **desktop** (3.96 % → 4.43 %, p < 0.01). At the same time, sign-up conversion in the **United States** dropped significantly (8.79 % → 7.93 %, p < 0.01). This is not visible in the total but is worth checking before rollout.
- **Test 2:** in **Canada**, `add_shipping_info` and `begin_checkout` decreased significantly (5.0 % → 3.6 %).
- Overall, 65 of 1,968 segment checks (≈ 3.3 %) were significant. That is below the 5 % expected by chance at α = 0.05, so isolated "significant" segments without confirmation in the total should be treated as hypotheses for further tests, not as proven effects.

### Summary
3 of the 16 overall checks were significant.
- **Test 1** is a candidate for rollout, after checking why the checkout gain does not carry through to payment and why sign-ups fall in the US.
- **Test 4** should not be rolled out because it reduces sign-ups.
- **Tests 2 and 3** showed no effect.

### Limitations
- Results in small segments are unreliable: a few dozen sessions can produce "significance" from 0 → 5 events.
- Testing many segments at once increases the chance of false positives (multiple comparisons). A Bonferroni or Benjamini–Hochberg correction would make the segment analysis stricter.